In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn
import torch
from torch.utils.data import TensorDataset, DataLoader

In [30]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [31]:
train_df.shape

(42000, 785)

In [32]:
X = train_df.drop("label", axis=1).values
y = train_df["label"].values

In [33]:
X = X/255
test_df = test_df.values/255

In [34]:
X = X.reshape(-1, 1, 28, 28)
test_df = test_df.reshape(-1, 1, 28, 28)

In [35]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.from_numpy(X_train).float()
X_val = torch.from_numpy(X_val).float()
y_train = torch.from_numpy(y_train).long().squeeze()
y_val = torch.from_numpy(y_val).long().squeeze()

In [36]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.Linear(128, 10),
        )

    def forward(self, X):
        return self.model(X)




In [37]:
model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [38]:

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

In [39]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    acc = correct / total
    print(f"Epoch {epoch} | Loss {loss:.4f} | Val Acc {acc:.4f}")

Epoch 0 | Loss 0.0310 | Val Acc 0.9724
Epoch 1 | Loss 0.0198 | Val Acc 0.9770
Epoch 2 | Loss 0.0147 | Val Acc 0.9799
Epoch 3 | Loss 0.0446 | Val Acc 0.9856
Epoch 4 | Loss 0.0082 | Val Acc 0.9849
Epoch 5 | Loss 0.0753 | Val Acc 0.9810
Epoch 6 | Loss 0.0013 | Val Acc 0.9850
Epoch 7 | Loss 0.0651 | Val Acc 0.9871
Epoch 8 | Loss 0.0188 | Val Acc 0.9867
Epoch 9 | Loss 0.0040 | Val Acc 0.9875


In [40]:
test_tensor = torch.tensor(test_df, dtype=torch.float32)
with torch.no_grad():
    logits = model(test_tensor)
    preds = torch.argmax(logits, dim=1)
submission = pd.DataFrame({
    "ImageId": range(1,len(preds)+1),
    "Label": preds
})
submission.to_csv("submission.csv", index=False)

In [41]:
# %%
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torchvision import transforms

# %%
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# %%
X = train_df.drop("label", axis=1).values / 255.0
y = train_df["label"].values

test_X = test_df.values / 255.0

# %%
X = X.reshape(-1, 1, 28, 28).astype("float32")
test_X = test_X.reshape(-1, 1, 28, 28).astype("float32")

# %%
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# %%
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).long()
X_val_tensor = torch.from_numpy(X_val)
y_val_tensor = torch.from_numpy(y_val).long()
test_tensor = torch.from_numpy(test_X)

# %%
# Data augmentation on training set
train_transforms = transforms.Compose(
    [
        transforms.RandomRotation(10),
        transforms.RandomAffine(0, translate=(0.1, 0.1)),
    ]
)


class AugmentedDataset(TensorDataset):
    def __init__(self, X, y, transform=None):
        super().__init__(X, y)
        self.transform = transform

    def __getitem__(self, idx):
        img, label = super().__getitem__(idx)
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset = AugmentedDataset(
    X_train_tensor, y_train_tensor, transform=train_transforms
)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)


# %%
class StrongCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.model(x)


# %%
device = "cuda" if torch.cuda.is_available() else "cpu"
model = StrongCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# %%
epochs = 15
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    val_acc = correct / total
    print(
        f"Epoch {epoch+1} | Train Loss {train_loss/len(train_loader):.4f} | Val Acc {val_acc:.4f}"
    )

# %%
# Inference
model.eval()
with torch.no_grad():
    test_tensor = test_tensor.to(device)
    logits = model(test_tensor)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

submission = pd.DataFrame({"ImageId": range(1, len(preds) + 1), "Label": preds})
submission.to_csv("submission.csv", index=False)

Epoch 1 | Train Loss 0.5903 | Val Acc 0.9708
Epoch 2 | Train Loss 0.2476 | Val Acc 0.9821
Epoch 3 | Train Loss 0.2065 | Val Acc 0.9830
Epoch 4 | Train Loss 0.1573 | Val Acc 0.9860
Epoch 5 | Train Loss 0.1388 | Val Acc 0.9889
Epoch 6 | Train Loss 0.1351 | Val Acc 0.9883
Epoch 7 | Train Loss 0.1146 | Val Acc 0.9915
Epoch 8 | Train Loss 0.1105 | Val Acc 0.9914
Epoch 9 | Train Loss 0.1091 | Val Acc 0.9915
Epoch 10 | Train Loss 0.0987 | Val Acc 0.9920
Epoch 11 | Train Loss 0.0974 | Val Acc 0.9914
Epoch 12 | Train Loss 0.0910 | Val Acc 0.9926
Epoch 13 | Train Loss 0.0863 | Val Acc 0.9919
Epoch 14 | Train Loss 0.0828 | Val Acc 0.9924
Epoch 15 | Train Loss 0.0834 | Val Acc 0.9923
